# 03 — FinBERT Sentiment Scoring (Colab GPU)

**Purpose:** Score 639K CEO-relevant Reddit comments using FinBERT for financial sentiment.

**Model:** `ProsusAI/finbert` — fine-tuned BERT for financial sentiment (positive/negative/neutral)

**Runtime:** Google Colab with T4 GPU (~1-3 hours)

**Before running:**
1. Upload `ceo_mentions_clean.parquet` (117 MB) to Google Drive at `ceo_reddit/data/`
2. Set Runtime > Change runtime type > T4 GPU

## Step 1 — Setup & GPU Check

In [ ]:
# Verify GPU is available
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    print("Running on CPU will take 40-80 hours instead of 1-3 hours.")

print(f"Using device: {device}")

In [ ]:
# Install dependencies (Colab has most pre-installed)
!pip install -q transformers fastparquet

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Data paths on Drive
DRIVE_DATA = Path("/content/drive/MyDrive/ceo_reddit/data")
INPUT_FILE = DRIVE_DATA / "ceo_mentions_clean.parquet"

assert INPUT_FILE.exists(), f"File not found: {INPUT_FILE}\nUpload ceo_mentions_clean.parquet to Google Drive at ceo_reddit/data/"
print(f"Input file: {INPUT_FILE} ({INPUT_FILE.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# Copy to local Colab disk for faster I/O (Drive is slow for random reads)
LOCAL_DIR = Path("/content/ceo_reddit")
LOCAL_DIR.mkdir(exist_ok=True)
LOCAL_INPUT = LOCAL_DIR / "ceo_mentions_clean.parquet"

if not LOCAL_INPUT.exists():
    import shutil
    shutil.copy2(INPUT_FILE, LOCAL_INPUT)
    print(f"Copied to local disk: {LOCAL_INPUT}")
else:
    print(f"Already on local disk: {LOCAL_INPUT}")

## Step 2 — Load Data

In [ ]:
import pandas as pd

df = pd.read_parquet(LOCAL_INPUT, engine="fastparquet")
print(f"Loaded: {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

# Filter to scorable comments (same as dictionary scorer)
valid_mask = (
    ~df["full_text"].isin(["[deleted]", "[removed]", ""])
    & (df["full_text"].str.len() >= 10)
    & ~df["author"].isin(["[deleted]", "[removed]"])
)
df = df[valid_mask].reset_index(drop=True)
print(f"After quality filter: {len(df):,} rows")
print(f"Median text length: {df['full_text'].str.len().median():.0f} chars")

## Step 3 — Load FinBERT Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

MODEL_NAME = "ProsusAI/finbert"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

# FinBERT label mapping
LABELS = ["positive", "negative", "neutral"]
print(f"Model loaded on {device}")
print(f"Labels: {LABELS}")
print(f"Max sequence length: {tokenizer.model_max_length}")

In [ ]:
# Quick test with sample texts
test_texts = [
    "The CEO delivered record earnings and raised guidance for next quarter.",
    "This CEO is destroying the company. Stock is crashing and employees are leaving.",
    "Tim Cook announced the new iPhone release date today.",
    "Elon Musk is delusional, Tesla is way overvalued.",
    "Warren Buffett is the greatest investor of all time.",
]

with torch.no_grad():
    inputs = tokenizer(test_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()

print(f"{'Text':<70} {'Pos':>6} {'Neg':>6} {'Neu':>6} {'Label':>10}")
print("-" * 100)
for text, prob in zip(test_texts, probs):
    label = LABELS[prob.argmax()]
    print(f"{text[:68]:<70} {prob[0]:>6.3f} {prob[1]:>6.3f} {prob[2]:>6.3f} {label:>10}")

## Step 4 — Batch Scoring

Process all 639K comments in batches of 32. Checkpoints every 50K rows to Google Drive so we can resume if the Colab session disconnects.

In [ ]:
import numpy as np
import time

BATCH_SIZE = 32
CHECKPOINT_EVERY = 50_000  # Save progress every 50K rows
CHECKPOINT_DIR = DRIVE_DATA / "finbert_checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Check if we have a partial checkpoint to resume from
checkpoint_file = CHECKPOINT_DIR / "finbert_scores_partial.parquet"
start_idx = 0

if checkpoint_file.exists():
    partial = pd.read_parquet(checkpoint_file, engine="fastparquet")
    start_idx = len(partial)
    print(f"Resuming from checkpoint: {start_idx:,} rows already scored")
    # Pre-fill results arrays from checkpoint
    all_positive = partial["finbert_positive"].tolist()
    all_negative = partial["finbert_negative"].tolist()
    all_neutral = partial["finbert_neutral"].tolist()
else:
    all_positive = []
    all_negative = []
    all_neutral = []

total = len(df)
remaining = total - start_idx
print(f"Total rows: {total:,}")
print(f"Remaining to score: {remaining:,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Estimated batches: {remaining // BATCH_SIZE:,}")

In [ ]:
# Main scoring loop
texts = df["full_text"].values
start_time = time.time()

for batch_start in range(start_idx, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)
    batch_texts = [str(t)[:2000] for t in texts[batch_start:batch_end]]  # Truncate very long texts

    with torch.no_grad():
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()

    all_positive.extend(probs[:, 0].tolist())
    all_negative.extend(probs[:, 1].tolist())
    all_neutral.extend(probs[:, 2].tolist())

    # Progress logging
    scored = batch_end
    if scored % 10_000 < BATCH_SIZE:
        elapsed = time.time() - start_time
        done_this_run = scored - start_idx
        rate = done_this_run / elapsed if elapsed > 0 else 0
        remaining_rows = total - scored
        eta_min = remaining_rows / rate / 60 if rate > 0 else 0
        print(f"  {scored:>7,} / {total:,} ({scored/total:.1%}) | {rate:.0f} rows/sec | ETA: {eta_min:.0f} min")

    # Checkpoint to Drive
    if scored % CHECKPOINT_EVERY < BATCH_SIZE and scored > start_idx:
        print(f"  Saving checkpoint at {scored:,} rows...")
        ckpt_df = df.iloc[:scored].copy()
        ckpt_df["finbert_positive"] = all_positive[:scored]
        ckpt_df["finbert_negative"] = all_negative[:scored]
        ckpt_df["finbert_neutral"] = all_neutral[:scored]
        ckpt_df.to_parquet(checkpoint_file, engine="fastparquet", compression="zstd", index=False)
        print(f"  Checkpoint saved: {checkpoint_file}")

elapsed = time.time() - start_time
done_this_run = total - start_idx
print(f"\nDone! Scored {done_this_run:,} rows in {elapsed/60:.1f} min ({done_this_run/elapsed:.0f} rows/sec)")

## Step 5 — Attach Scores & Derive Traits

In [ ]:
# Attach FinBERT scores to dataframe
df["finbert_positive"] = np.array(all_positive, dtype=np.float32)
df["finbert_negative"] = np.array(all_negative, dtype=np.float32)
df["finbert_neutral"] = np.array(all_neutral, dtype=np.float32)

# Derived scores
df["finbert_label"] = df[["finbert_positive", "finbert_negative", "finbert_neutral"]].idxmax(axis=1).str.replace("finbert_", "")
df["finbert_sentiment"] = df["finbert_positive"] - df["finbert_negative"]  # Range: -1 to +1
df["finbert_confidence"] = df[["finbert_positive", "finbert_negative", "finbert_neutral"]].max(axis=1)

print(f"Rows scored: {len(df):,}")
print(f"\nLabel distribution:")
print(df["finbert_label"].value_counts().to_string())
print(f"\nSentiment score stats:")
print(df["finbert_sentiment"].describe().to_string())

In [ ]:
# Quick sanity check — sample texts by sentiment
print("=== MOST POSITIVE (FinBERT) ===")
for _, r in df.nlargest(5, "finbert_sentiment").iterrows():
    print(f"  [{r['finbert_sentiment']:+.3f}] [{r['ceo_matched']}] {r['full_text'][:150]}")

print(f"\n=== MOST NEGATIVE (FinBERT) ===")
for _, r in df.nsmallest(5, "finbert_sentiment").iterrows():
    print(f"  [{r['finbert_sentiment']:+.3f}] [{r['ceo_matched']}] {r['full_text'][:150]}")

print(f"\n=== MOST NEUTRAL (FinBERT) ===")
for _, r in df.nlargest(5, "finbert_neutral").iterrows():
    print(f"  [neu={r['finbert_neutral']:.3f}] [{r['ceo_matched']}] {r['full_text'][:150]}")

## Step 6 — CEO-Year Aggregates

In [ ]:
# Aggregate to CEO-year level
ceo_year = df.groupby(["execid", "ceo_matched", "company_matched", "ticker_matched", "year"]).agg(
    mention_count=("comment_id", "count"),
    finbert_sentiment_mean=("finbert_sentiment", "mean"),
    finbert_sentiment_median=("finbert_sentiment", "median"),
    finbert_sentiment_std=("finbert_sentiment", "std"),
    finbert_positive_mean=("finbert_positive", "mean"),
    finbert_negative_mean=("finbert_negative", "mean"),
    finbert_neutral_mean=("finbert_neutral", "mean"),
    pct_positive=("finbert_label", lambda x: (x == "positive").mean()),
    pct_negative=("finbert_label", lambda x: (x == "negative").mean()),
    pct_neutral=("finbert_label", lambda x: (x == "neutral").mean()),
    avg_confidence=("finbert_confidence", "mean"),
).reset_index()

print(f"CEO-year rows: {len(ceo_year):,}")
ceo_year.head()

In [ ]:
# Top CEOs by sentiment (min 100 mentions)
ceo_agg = df.groupby(["execid", "ceo_matched", "company_matched"]).agg(
    mentions=("comment_id", "count"),
    avg_sentiment=("finbert_sentiment", "mean"),
    pct_positive=("finbert_label", lambda x: (x == "positive").mean()),
    pct_negative=("finbert_label", lambda x: (x == "negative").mean()),
).reset_index()

sig = ceo_agg[ceo_agg["mentions"] >= 100]

print(f"\n=== MOST NEGATIVE CEOs (FinBERT, min 100 mentions) ===")
for _, r in sig.nsmallest(15, "avg_sentiment").iterrows():
    print(f"  {r['ceo_matched']:<35} {r['company_matched']:<25} sent={r['avg_sentiment']:+.4f}  neg={r['pct_negative']:.1%}  ({r['mentions']:,})")

print(f"\n=== MOST POSITIVE CEOs (FinBERT, min 100 mentions) ===")
for _, r in sig.nlargest(15, "avg_sentiment").iterrows():
    print(f"  {r['ceo_matched']:<35} {r['company_matched']:<25} sent={r['avg_sentiment']:+.4f}  pos={r['pct_positive']:.1%}  ({r['mentions']:,})")

In [ ]:
# Elon Musk sentiment over time (sanity check)
musk = df[df["ceo_matched"] == "Elon R. Musk"].groupby("year").agg(
    mentions=("comment_id", "count"),
    sentiment=("finbert_sentiment", "mean"),
    pct_neg=("finbert_label", lambda x: (x == "negative").mean()),
).reset_index()

print("Elon Musk FinBERT sentiment by year:")
for _, r in musk.iterrows():
    bar = "+" * max(0, int(r["sentiment"] * 50)) + "-" * max(0, int(-r["sentiment"] * 50))
    print(f"  {r['year']:.0f}: {r['sentiment']:+.4f} ({r['mentions']:>6,.0f} mentions, {r['pct_neg']:.0%} neg) {bar}")

## Step 7 — Save Results

In [ ]:
# Save full scored dataset to Drive
output_file = DRIVE_DATA / "ceo_mentions_finbert_scored.parquet"
df.to_parquet(output_file, engine="fastparquet", compression="zstd", index=False)
print(f"Saved: {output_file} ({output_file.stat().st_size / 1024 / 1024:.1f} MB)")

# Save CEO-year aggregates
agg_file = DRIVE_DATA / "ceo_year_finbert_scores.csv"
ceo_year.to_csv(agg_file, index=False)
print(f"Saved: {agg_file} ({len(ceo_year):,} rows)")

# Clean up checkpoint
if checkpoint_file.exists():
    checkpoint_file.unlink()
    print("Removed checkpoint file (scoring complete).")

In [ ]:
# Download CSVs locally
from google.colab import files

# Download the CEO-year aggregate (small, useful for quick analysis)
files.download(str(agg_file))

## Step 8 — Summary

In [ ]:
print("=" * 60)
print("FINBERT SCORING COMPLETE")
print("=" * 60)
print(f"")
print(f"Comments scored:     {len(df):,}")
print(f"Unique CEOs:         {df['execid'].nunique()}")
print(f"CEO-year combos:     {len(ceo_year):,}")
print(f"")
print(f"Label distribution:")
for label in ['positive', 'negative', 'neutral']:
    pct = (df['finbert_label'] == label).mean()
    print(f"  {label:<10}: {pct:.1%}")
print(f"")
print(f"Sentiment score:")
print(f"  Mean:   {df['finbert_sentiment'].mean():+.4f}")
print(f"  Median: {df['finbert_sentiment'].median():+.4f}")
print(f"  Std:    {df['finbert_sentiment'].std():.4f}")
print(f"")
print(f"Outputs on Drive:")
print(f"  {output_file}")
print(f"  {agg_file}")
print(f"")
print(f"Next steps:")
print(f"  1. Download scored parquet from Drive to local machine")
print(f"  2. Compare FinBERT vs dictionary scores (cross-method validation)")
print(f"  3. LLM labeling of 5-10K sample for ground truth")
print(f"  4. Score earnings call transcripts with same FinBERT model")
print(f"  5. Compute self-presentation discrepancy per CEO-quarter")